# 08 Style LLM Training

Goal: train the style injection weights so a frozen base LLM learns:

```text
style-regenerated draft + author style embedding -> real author passage
```

This is the first Style LLM training stage. It trains only the style projection adapter, not the base model.

In [ ]:
import os
import random
import sys
from itertools import cycle
from pathlib import Path

def find_project_root():
    env_root = os.getenv("VOICE_CLONE_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if (candidate / "voice_clone").exists():
            return candidate
    raise RuntimeError("Could not find project root. Set VOICE_CLONE_ROOT to the repo path.")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from voice_clone.data.dataset import triplet_from_row
from voice_clone.data.io import read_jsonl
from voice_clone.models import StyleConditionedCausalLM, count_parameters

print("ROOT:", ROOT)

In [ ]:
TRIPLETS_PATH = ROOT / "data" / "processed" / "triplets_style_regen_small.jsonl"
AUTHOR_EMBEDDINGS_PATH = ROOT / "data" / "processed" / "author_style_embeddings.pt"
CHECKPOINT_DIR = ROOT / "checkpoints"

BASE_MODEL = "distilgpt2"
# BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

SEED = 7
MAX_LENGTH = 768
TARGET_CHARS = 1200
DRAFT_CHARS = 1200
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4
STEPS = 500
LR = 3e-4
WARMUP_STEPS = 50
LOG_EVERY = 10
SAVE_EVERY = 100

random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else None

print("device:", device)
print("triplets:", TRIPLETS_PATH)
print("author embeddings:", AUTHOR_EMBEDDINGS_PATH)

In [ ]:
author_payload = torch.load(AUTHOR_EMBEDDINGS_PATH, map_location="cpu")
author_embeddings = author_payload["author_embeddings"]

records = [triplet_from_row(row) for row in read_jsonl(TRIPLETS_PATH)]
records = [r for r in records if r.author_id in author_embeddings and (r.style_regenerated_draft or r.neutral_draft)]

print("records:", len(records))
print("authors:", sorted({r.author_id for r in records}))
print("negative type:", records[0].negative_type)
print("draft sample:", (records[0].style_regenerated_draft or records[0].neutral_draft)[:300])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = StyleConditionedCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
).to(device)
model.train()

if hasattr(model.base_model, "gradient_checkpointing_enable"):
    try:
        model.base_model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
    except TypeError:
        model.base_model.gradient_checkpointing_enable()
if hasattr(model.base_model, "enable_input_require_grads"):
    model.base_model.enable_input_require_grads()
if hasattr(model.base_model.config, "use_cache"):
    model.base_model.config.use_cache = False

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
print(count_parameters(model))

In [ ]:
def clip_text(text, chars):
    return " ".join(text[:chars].split())


def make_prompt(draft):
    return (
        "Rewrite the draft below so it matches the target author's style while preserving meaning.\n\n"
        "DRAFT:\n"
        f"{draft}\n\n"
        "REWRITE:\n"
    )


def collate(batch):
    prompts = []
    targets = []
    style_embeddings = []
    for record in batch:
        draft = clip_text(record.style_regenerated_draft or record.neutral_draft, DRAFT_CHARS)
        target = clip_text(record.real_passage, TARGET_CHARS)
        prompts.append(make_prompt(draft))
        targets.append(target + tokenizer.eos_token)
        style_embeddings.append(author_embeddings[record.author_id])
    return prompts, targets, torch.stack(style_embeddings)


def encode_batch(prompts, targets):
    full_texts = [p + t for p, t in zip(prompts, targets)]
    prompt_ids = tokenizer(prompts, add_special_tokens=False)["input_ids"]
    encoded = tokenizer(
        full_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    labels = encoded["input_ids"].clone()
    for i, ids in enumerate(prompt_ids):
        labels[i, : min(len(ids), labels.shape[1])] = -100
    labels[encoded["attention_mask"] == 0] = -100
    encoded = {key: value.to(device) for key, value in encoded.items()}
    return encoded, labels.to(device)


loader = DataLoader(records, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
batch = next(iter(loader))
encoded, labels = encode_batch(batch[0], batch[1])
print({k: tuple(v.shape) for k, v in encoded.items()})
print("supervised tokens:", int((labels != -100).sum()))

## Tiny Overfit Test

This uses one batch and should generally push loss downward. It is a mechanics test, not quality evaluation.

In [ ]:
test_prompts, test_targets, test_styles = batch
test_styles = test_styles.to(device)
encoded, labels = encode_batch(test_prompts, test_targets)

model.train()
losses = []
for i in range(10):
    out = model(**encoded, labels=labels, style_embedding=test_styles)
    loss = out.loss
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
    optimizer.step()
    losses.append(loss.item())
    print(f"overfit_step={i+1} loss={loss.item():.4f}")

print("initial/final:", losses[0], losses[-1])

## Training Loop

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
train_iter = cycle(loader)
optimizer.zero_grad(set_to_none=True)

warmup_optimizer_steps = max(1, WARMUP_STEPS // GRAD_ACCUM_STEPS)
def lr_lambda(optimizer_step):
    return min(1.0, (optimizer_step + 1) / warmup_optimizer_steps)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

ema_loss = None
ema_alpha = 0.05

for step in range(1, STEPS + 1):
    prompts, targets, styles = next(train_iter)
    styles = styles.to(device)
    encoded, labels = encode_batch(prompts, targets)

    out = model(**encoded, labels=labels, style_embedding=styles)
    loss = out.loss
    (loss / GRAD_ACCUM_STEPS).backward()

    if step % GRAD_ACCUM_STEPS == 0:
        grad_norm = torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0
        )
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

    loss_val = loss.item()
    ema_loss = loss_val if ema_loss is None else ema_alpha * loss_val + (1 - ema_alpha) * ema_loss

    if step % LOG_EVERY == 0 or step == 1:
        current_lr = optimizer.param_groups[0]["lr"]
        gnorm = grad_norm.item() if step % GRAD_ACCUM_STEPS == 0 else float("nan")
        print(f"step={step:04d} loss={loss_val:.4f} ema={ema_loss:.4f} gnorm={gnorm:.3f} lr={current_lr:.2e}")

    if step % SAVE_EVERY == 0 or step == STEPS:
        tmp_path = CHECKPOINT_DIR / "style_llm_stage1.tmp.pt"
        checkpoint_path = CHECKPOINT_DIR / "style_llm_stage1.pt"
        torch.save(
            {
                "base_model": BASE_MODEL,
                "style_projection_state_dict": model.style_projection.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "step": step,
                "config": {
                    "max_length": MAX_LENGTH,
                    "draft_chars": DRAFT_CHARS,
                    "target_chars": TARGET_CHARS,
                    "style_dim": 512,
                },
            },
            tmp_path,
        )
        tmp_path.replace(checkpoint_path)
        print(f"saved: {checkpoint_path}  (ema={ema_loss:.4f})")